In [ ]:
from detectron2.utils.logger import setup_logger

import gc
import os
import time
import sys
import json
# allows us to import from the custom configs directory w/o affecting deepdisc library imports
sys.path.insert(0, '/work/hdd/bdsp/g4merz/RubinRoman/configs')
from custom.trainers import return_timed_lazy_trainer
from custom.hooks import return_timed_evallossHook
#from custom.meta_arch import return_frozen_model, return_unfrozen_model

import detectron2.utils.comm as comm

# import some common libraries
import numpy as np
import torch
import torch.distributed as dist

# import some common detectron2 utilities
from detectron2.config import LazyConfig
from detectron2.engine import launch

# Note: hsc_test_augs and train_augs don't exist in deepdisc
from deepdisc.data_format.image_readers import wlDC2ImageReader, wlHSCImageReader
from deepdisc.data_format.register_data import register_data_set
from deepdisc.model.loaders import return_test_loader, return_train_loader
from deepdisc.model.models import return_lazy_model
from deepdisc.utils.parse_arguments import make_training_arg_parser
#from custom.meta_arch import return_frozen_model, return_unfrozen_model
import ssl
import torch.nn as nn
import matplotlib.pyplot as plt
from deepdisc.inference.predictors import AstroPredictor
from deepdisc.data_format.file_io import get_data_from_json
import torch.nn.functional as F
import random

from astropy.wcs import WCS


In [ ]:
data_root_dir = '/work/hdd/bdsp/yse2/lsst_data/'
anns_folder = 'annotations_lvl5'
test_data_fn = f'{data_root_dir}{anns_folder}/test_8k_keypoints_wcs.json'
#test_data_fn = f'{data_root_dir}{anns_folder}/test_keypoints.json'
eval_data_fn = f'{data_root_dir}{anns_folder}/val_4k_keypoints_wcs.json'

base_run_dir = '/work/hdd/bdsp/yse2/lsst_runs/'
run_name = 'clip5_30k_4h200_bs192_ep15_lprj'
#run_name = 'comb_30k_4h200_bs144_ep50'

#run_name = 'clip5_30k_4h200_bs64_ep50'
#run_name = 'lsst5_30k_4h200_bs192_ep50'

run_dir = f'{base_run_dir}{run_name}'
model_path = f'{run_dir}/{run_name}.pth'
#model_path = f'{run_dir}/checkpoints/clip_30k_4h200_bs64_ep50_new_1.pth'



In [ ]:
#cfgfile = "/work/hdd/bdsp/g4merz/RubinRoman/configs/solo/swin_comb_job_keypoint_30k.py"
#cfgfile = "/work/hdd/bdsp/g4merz/RubinRoman/configs/solo/swin_distill_job_keypoint_30k.py"
cfgfile = "/work/hdd/bdsp/g4merz/RubinRoman/configs/solo/swin_clip_lsst_roman_30k.py"


bs=1
cfg = LazyConfig.load(cfgfile)
cfg.train.device='cpu'
cfg.train.init_checkpoint = model_path
cfg.SEED = -1
cfg.dataloader.train.total_batch_size = bs
cfg.SOLVER.IMS_PER_BATCH = bs
#cfg.dataloader.augs=None
cfg.dataloader.train.keypoint_hflip_indices=[0]
cfg.dataloader.test.keypoint_hflip_indices=[0]

for box_predictor in cfg.model.roi_heads.box_predictors:
    box_predictor.test_topk_per_image = 2000
    box_predictor.test_score_thresh = 0.5
    box_predictor.test_nms_thresh = 0.7
predictor = AstroPredictor(cfg)


In [ ]:
model = predictor.model

In [ ]:
from detectron2.data import MetadataCatalog, DatasetCatalog

custom_colors = [
    (0, 255, 0),    # green for galaxies
    (0, 0, 255),    # blue for stars
]

custom_prop_colors = [
    (255,255,255), # white for foreground for background
    (255, 0, 0)    
    
]

if cfg.DATASETS.TEST in MetadataCatalog:
    DatasetCatalog.remove(cfg.DATASETS.TEST)
    MetadataCatalog.remove(cfg.DATASETS.TEST)

if cfg.DATASETS.TRAIN in MetadataCatalog:
    DatasetCatalog.remove(cfg.DATASETS.TRAIN)
    MetadataCatalog.remove(cfg.DATASETS.TRAIN)
    
if "astro_prop" in MetadataCatalog:
    DatasetCatalog.remove("astro_prop")
    MetadataCatalog.remove("astro_prop")

astrotest_metadata = register_data_set(cfg.DATASETS.TEST, test_data_fn, thing_classes=cfg.metadata.classes, thing_colors=custom_colors)
astrotest_metadata = register_data_set(cfg.DATASETS.TRAIN, eval_data_fn, thing_classes=cfg.metadata.classes, thing_colors=custom_colors)

astroprop_metadata = register_data_set("astro_prop", test_data_fn, thing_classes=["background","foreground"], thing_colors=custom_prop_colors)

In [ ]:
import os
def lsst_key_mapper(dataset_dict):
    key = dataset_dict["file_name"]
    im = key.split('truth/')[1]
    k = os.path.join("/work/hdd/bdsp/yse2/lsst_data/truth/",im)
    return k

#mapper = cfg.dataloader.test.mapper(cfg.dataloader.test.keypoint_hflip_indices,cfg.dataloader.test.cache_dir,
#                                     cfg.dataloader.test.imagereader, lsst_key_mapper, None).map_data # No augs for test set
#mapper = cfg.dataloader.test.mapper(cfg.dataloader.test.keypoint_hflip_indices,
#                                     cfg.dataloader.test.imagereader, lsst_key_mapper, None).map_data # No augs for test set

mapper = cfg.dataloader.train.mapper(cfg.dataloader.train.keypoint_hflip_indices,
                                     cfg.dataloader.train.imagereader, lsst_key_mapper, None).map_data # No augs for test set

#test_loader = return_test_loader(cfg, mapper)
train_loader = return_train_loader(cfg, mapper)


In [ ]:
test_dicts = get_data_from_json(test_data_fn)

In [ ]:
#b = next(iter(train_loader))
i1 = np.random.randint(len(test_dicts))
i2=i1
while i2==i1:
    i2 = np.random.randint(len(test_dicts))
dm1 = [mapper(test_dicts[i1])]
dm2 = [mapper(test_dicts[i2])]

In [ ]:
i1,i2

In [ ]:
unrec_blends = get_data_from_json('/work/hdd/bdsp/yse2/lsst_runs/lsst5_30k_4h200_bs192_ep50/unrec_blends/unrec_blend_imgs_ll1.0_s0.4_n0.55.json')
def unrec_blend_sample(k):
    random_blends = random.sample(unrec_blends['selected_file_names'],k=k)
    inds=[]
    for i,d in enumerate(test_dicts):
        for rb in random_blends:
            if rb in d['file_name']:
                inds.append(i)
    return inds

#b = [mapper(test_dicts[3101])]

In [ ]:
from detectron2.modeling.roi_heads import CascadeROIHeads, select_foreground_proposals
from detectron2.modeling.proposal_generator.proposal_utils import add_ground_truth_to_proposals


def get_embeddings(proposal_option='all',pos_thresh=0.5):
    #for moco
    with torch.no_grad():
        images_q = model.preprocess_image(dm1, "image_rubin", model.rubin_pixel_mean, 
                                                model.rubin_pixel_std, model.backbone_q.size_divisibility, 
                                                model.backbone_q.padding_constraints)
                
        features_q = model.backbone_q(images_q.tensor)
        #q = model.moco.mlp_q(features_q)
        #q = nn.functional.normalize(q, dim=1)
        
        images_k = model.preprocess_image(dm2, "image_roman", model.roman_pixel_mean, 
                                                model.roman_pixel_std, model.backbone_k.size_divisibility, 
                                                model.backbone_k.padding_constraints)
    
        features_k = model.backbone_k(images_k.tensor)  # keys: NxC
    
    
        proposals, _ = model.proposal_generator(images_q, features_q, None)
    
        pred_instances = model.roi_heads._forward_box(features_q, proposals)
        # mask + keypoint heads: run on surviving boxes after NMS
        pred_instances = model.roi_heads.forward_with_given_boxes(features_q, pred_instances)
    
        for p in proposals:
            p.pred_boxes = p.proposal_boxes
    
        proposals_pos =[]
        proposals_neg =[]
    
        for p in proposals:
            pos = np.where(torch.sigmoid(p.objectness_logits)>pos_thresh)
            neg = np.where(torch.sigmoid(p.objectness_logits)<pos_thresh)
            proposals_pos.append(p[pos])
            proposals_neg.append(p[neg])

        if proposal_option=='all':
            props = proposals
        elif proposal_option=='positive':
            props = proposals_pos
        elif proposal_option=='negative':
            props = proposals_neg
        elif proposal_option=='predicted':
            props = pred_instances
        
        
        zq,zk = model.roi_heads.forward_contrastive(
                    features_q, 
                    features_k, 
                    props,
                    targets=None
                )            
    
    return zq,zk,props


In [ ]:
zq_pos,zk_pos,props_pos = get_embeddings('positive')
logits_pos = (zq_pos @ zq_pos.T) * model.roi_heads.logit_scale.exp()

zq_neg,zk_neg,props_neg = get_embeddings('negative')
logits_neg = (zq_neg @ zq_neg.T) * model.roi_heads.logit_scale.exp()

zq,zk,props = get_embeddings('predicted')
logits = (zq @ zq.T) * model.roi_heads.logit_scale.exp()

zq_all,zk_all,props_all = get_embeddings('all')
logits_all = (zq_all @ zq_all.T) * model.roi_heads.logit_scale.exp()

In [ ]:
#zq,zk,props = get_embeddings('positive')
#logits_qk = torch.bmm(zq, zk.transpose(1, 2)) * model.roi_heads.logit_scale.exp()  # (N, L*49, L*49)
#logits_qq = torch.bmm(zq, zq.transpose(1, 2)) * model.roi_heads.logit_scale.exp()  # (N, L*49, L*49)


zq,zk,props = get_embeddings('negative')
#zq,zk,props = get_embeddings('predicted')
logits_qk = (zq @ zk.T) * model.roi_heads.logit_scale.exp()
logits_qq = (zq @ zq.T) * model.roi_heads.logit_scale.exp()


In [ ]:
fig,ax = plt.subplots(1,2,figsize=(10,5))
#m = 110
#ax[0].imshow(logits_qk[m])
#ax[1].imshow(logits_qq[m])


ax[0].imshow(logits_qk)
ax[1].imshow(logits_qq)

ax[0].set_ylabel('LSST boxes')
ax[0].set_xlabel('Roman boxes')
ax[0].set_title('zq zk logits (similarity)')


ax[1].set_ylabel('LSST boxes')
ax[1].set_xlabel('LSST boxes')
ax[1].set_title('zq zq logits (similarity)')

In [ ]:
#Look at UMAP clustering of latent space old vs new
#import umap


In [ ]:
#b = next(iter(train_loader))
i1 = np.random.randint(len(test_dicts))
i2=i1
while i2==i1:
    i2 = np.random.randint(len(test_dicts))
dm1 = [mapper(test_dicts[i1])]
dm2 = [mapper(test_dicts[i2])]

In [ ]:
import time
from deepdisc.astrodet.visualizer import Visualizer
from deepdisc.astrodet.visualizer import ColorMode
from detectron2 import structures
from astropy.visualization import make_lupton_rgb


image = dm1[0]['image_rubin']
b1 = image[3]
b2 = image[2]
b3 = image[1]
img = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=0.5, Q=5)


image2 = dm2[0]['image_roman']
b1 = image2[2]
b2 = image2[1]
b3 = image2[0]
img2 = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=15, Q=5)

scale=3.45
v0 = Visualizer(
    img,
    metadata=astrotest_metadata,
    scale=scale,
    instance_mode=ColorMode.SEGMENTATION,  # remove the colors of unsegmented pixels. This option is only available for segmentation models
)
groundTruth = v0.draw_dataset_dict(test_dicts[i1], lf=False, alpha=0.01)

#plt.figure(figsize=(12,6))

fig,ax = plt.subplots(1,2,figsize=(10,5),sharey=True)
#ax0 = plt.subplot(1, 2, 1)
ax0 = ax[0]

ax0.imshow(groundTruth.get_image())
#ax0.imshow(img)

ax0.axis("off")
ax0.set_title('Rubin Image')
t0 = time.time()
print(time.time()-t0, 'seconds')

#ax1 = plt.subplot(1, 2, 2)
ax2 = ax[1]
#ax2.imshow(out.get_image())
ax2.imshow(img2)
ax2.axis("off")
ax2.set_title('Roman Image')


#ax2.scatter(cs[:,0],cs[:,1],marker='x',color='purple',s=70)



plt.tight_layout()

plt.show()

In [ ]:
'''
lmax_qk = []
for i,l in enumerate(logits_qk[0]):
    lmax_qk.append(l.argmax().numpy())
lmax_qk = np.array(lmax_qk)

lmax_qq = []
for i,l in enumerate(logits_qq[0]):
    lmax_qq.append(l.argmax().numpy())
lmax_qq = np.array(lmax_qq)
'''

lmax_qk = []
for i,l in enumerate(logits_qk):
    lmax_qk.append(l.argmax().numpy())
lmax_qk = np.array(lmax_qk)

lmax_qq = []
for i,l in enumerate(logits_qq):
    lmax_qq.append(l.argmax().numpy())
lmax_qq = np.array(lmax_qq)


In [ ]:
fig,ax = plt.subplots(1,2,figsize=(10,5))
ax[0].scatter(np.arange(len(lmax_qk)),lmax_qk)
ax[0].plot((0,len(lmax_qk)),(0,len(lmax_qk)),color='k',linestyle=':')
ax[0].set_xlabel('zq index')
ax[0].set_ylabel('highest zk index')
ax[1].scatter(np.arange(len(lmax_qq)),lmax_qq)
ax[1].plot((0,len(lmax_qq)),(0,len(lmax_qq)),color='k',linestyle=':')
ax[1].set_xlabel('zq index')
ax[1].set_ylabel('highest zq index')

In [ ]:
#Look at logit matrix - strong diag is good
plt.imshow(logits_all)

In [ ]:
#take a look at closely overlapping boxes - use pairwise IOU and get logit matrix entries for a single object (row)

In [ ]:
from detectron2.structures import pairwise_iou

In [ ]:
ious = pairwise_iou(props[0].pred_boxes,props[0].pred_boxes)
ious_pos = pairwise_iou(props_pos[0].pred_boxes,props_pos[0].pred_boxes)
ious_neg = pairwise_iou(props_neg[0].pred_boxes,props_neg[0].pred_boxes)
ious_all = pairwise_iou(props_all[0].pred_boxes,props_all[0].pred_boxes)

In [ ]:
import matplotlib.colors as colors
def plot_logits(logits,ious):
    ll=0
    lu = logits.shape[0]
    #Look at a chunk of the logit matrix
    fig,ax = plt.subplots(1,2,figsize=(10,5))
    ax[0].imshow(logits[ll:lu,ll:lu],cmap='plasma')
    ax[1].imshow(ious[ll:lu,ll:lu],cmap='plasma',norm=colors.PowerNorm(gamma=0.5))
    #ax[1].imshow(ious[ll:lu,ll:lu],cmap='plasma',norm=colors.BoundaryNorm())

    ax[1].set_title('IOU')
    ax[0].set_title('Logits')

In [ ]:
plot_logits(logits_all,ious_all)
plt.suptitle('All proposals')

In [ ]:
plot_logits(logits_pos,ious_pos)
plt.suptitle('Foreground Proposals')

In [ ]:
plot_logits(logits_neg,ious_neg)
plt.suptitle('Background Proposals')

In [ ]:
plot_logits(logits,ious)
plt.suptitle('Predictions')

## Feature Map Visualization

In [ ]:
inds1 = [430,431,440]
inds2 = [610,611,642]
inds1 = random.sample(list(np.arange(len(test_dicts))),k=3)

dq = [mapper(test_dicts[i]) for i in inds1]
dk = [mapper(test_dicts[i]) for i in inds1]

#d2 = mapper(test_dicts[j])

In [ ]:
#no moco
with torch.no_grad():
    images_q = model.preprocess_image(dq, "image_rubin", model.rubin_pixel_mean, 
                                            model.rubin_pixel_std, model.backbone_q.size_divisibility, 
                                            model.backbone_q.padding_constraints)
    features_q = model.backbone_q(images_q.tensor)

    images_k = model.preprocess_image(dq, "image_roman", model.roman_pixel_mean, 
                                            model.roman_pixel_std, model.backbone_k.size_divisibility, 
                                            model.backbone_k.padding_constraints)

    features_k = model.backbone_k(images_k.tensor)  # keys: NxC


In [ ]:
fig,ax = plt.subplots(2,3,figsize=(9,6))
for i in range(3):
    ax[0][i].imshow(images_q.tensor[i][3].detach(),norm='asinh')
    ax[1][i].imshow(torch.sum(features_q['p2'],dim=1)[i].detach())
    #ax[1][i].imshow(features_q['p2'][i][203].detach())

plt.suptitle('With contrastive Loss')
#plt.savefig('/work/hdd/bdsp/g4merz/RubinRoman/plots/image_feature_map_p2_nocl.png')

In [ ]:
16000*0.07

In [ ]:
fig,ax = plt.subplots(2,3,figsize=(9,6))
for i in range(3):
    ax[0][i].imshow(images_q.tensor[i][3].detach(),norm='asinh')
    ax[1][i].imshow(images_k.tensor[i][3].detach(),norm='asinh')


In [ ]:
fig,ax = plt.subplots(2,3,figsize=(9,6))
for i in range(3):
    ax[0][i].imshow(torch.sum(features_q['p2'],dim=1)[i].detach())
    ax[1][i].imshow(torch.sum(features_k['p2'],dim=1)[i].detach())


## Model Visualization

In [ ]:
tfb = 'truth_'+os.path.basename(di['file_name']).replace('.npy','.json')
tfp = os.path.join(os.path.dirname(di['file_name'].replace('/u/yse2/lsst_data/truth/','/work/hdd/bdsp/yse2/lsst_data/truth/')),fb)
truth_cat = get_data_from_json(tfp)


In [ ]:
import time
from deepdisc.astrodet.visualizer import Visualizer
from deepdisc.astrodet.visualizer import ColorMode
from detectron2 import structures
from astropy.visualization import make_lupton_rgb

def plot_images(d,ddict,outputs,roman=False,plot_truth=False,rescale=False,buffer=1):

    if roman:
        image = d['image_rubin']
    else:
        image = d['image']
    
    b1 = image[3]
    b2 = image[2]
    b3 = image[1]

    img = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=0.5, Q=5)

    scale=1
    v0 = Visualizer(
        img,
        metadata=astrotest_metadata,
        scale=scale,
        instance_mode=ColorMode.SEGMENTATION,  # remove the colors of unsegmented pixels. This option is only available for segmentation models
    )
    groundTruth = v0.draw_dataset_dict(ddict, lf=False, alpha=0.01)

    #plt.figure(figsize=(12,6))

    fig,ax = plt.subplots(1,2,figsize=(10,5),sharey=True)
    #ax0 = plt.subplot(1, 2, 1)
    ax0 = ax[0]

    #ax0.imshow(groundTruth.get_image())
    ax0.imshow(img)

    if plot_truth:
        tfb = 'truth_'+os.path.basename(ddict['file_name']).replace('.npy','.json')
        tfp = os.path.join(os.path.dirname(ddict['file_name'].replace('/u/yse2/lsst_data/truth/','/work/hdd/bdsp/yse2/lsst_data/truth/')),tfb)
        truth_cat = get_data_from_json(tfp)
        t=0
        for obj in truth_cat:
            t+=1
            if obj['mag_i']<=25.3+buffer:
                if rescale:
                    sc = WCS(ddict['wcs']).pixel_to_world(obj['cutout_x'],obj['cutout_y'])
                    x_new,y_new = WCS(ddict['wcs_roman']).world_to_pixel(sc)
                    ax[0].scatter(x_new,y_new,color='white',s=30)
                else:
                    ax[0].scatter(obj['cutout_x'],obj['cutout_y'],color='white',s=30)
        print("total truth (with buffer): ", t)

    ax0.axis("off")
    ax0.set_title('Image')
    t0 = time.time()
    print(time.time()-t0, 'seconds')

    v2 = Visualizer(
            img,
            metadata=astrotest_metadata,
            scale=scale,
            instance_mode=ColorMode.SEGMENTATION, # remove the colors of unsegmented pixels. This option is only available for segmentation models
            keypoint_threshold=0
        )


    out = v2.draw_instance_predictions(outputs['instances'].to("cpu"),lf=False,ls='-',alpha=0.1,boxf=True, custom_labels=np.arange(len(outputs['instances'])))
    print("detected instances:", len(outputs['instances'].pred_boxes))
    print("")


    #ax1 = plt.subplot(1, 2, 2)
    ax2 = ax[1]
    ax2.imshow(out.get_image())
    ax2.axis("off")
    ax2.set_title('Detections')

    cs= outputs['instances'].pred_boxes.get_centers().detach().numpy()
    #ax2.scatter(cs[:,0],cs[:,1],marker='x',color='purple',s=70)


    plt.tight_layout()

    plt.show()

In [ ]:
inds = unrec_blend_sample(10)
inds

In [ ]:
test_ind = 1821
#test_ind = 6781 
di = test_dicts[test_ind]
mdi = mapper(di)
outputs = model([mdi])
plot_images(mdi,di,outputs[0],roman=True,plot_truth=True,rescale=False,buffer=1)

In [ ]:
uinds = unrec_blend_sample(10)
uinds

In [ ]:
#test_ind = 3101
test_ind = 2535 
di = test_dicts[test_ind]
mdi = mapper(di)
outputs = model([mdi])
plot_images(mdi,di,outputs[0],roman=True,plot_truth=True,buffer=1)

In [ ]:
import time
from deepdisc.astrodet.visualizer import Visualizer
from deepdisc.astrodet.visualizer import ColorMode
from detectron2 import structures
from astropy.visualization import make_lupton_rgb
from skimage.transform import rescale, resize, downscale_local_mean

def plot_images_romancomp(d1,d2,ddict,outputs,roman=False):

    fig,ax = plt.subplots(1,2,figsize=(10,5),sharey=True)
    ax0 = ax[0]
    ax1 = ax[1]


    image_rubin = d1['image_rubin']
    b1 = image_rubin[3]
    b2 = image_rubin[2]
    b3 = image_rubin[1]
    img_rubin = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=0.5, Q=5)


    image_roman = d2['image_roman']
    b1 = image_roman[2]
    b2 = image_roman[1]
    b3 = image_roman[0]
    b1 = resize(b1, (md1['height'],md1['width']), anti_aliasing=False)
    b2 = resize(b2, (md1['height'],md1['width']), anti_aliasing=False)
    b3 = resize(b3, (md1['height'],md1['width']), anti_aliasing=False)
    img_roman = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=15, Q=5)
    #img_roman = resize(img_roman, (d1['height'],d1['width']), anti_aliasing=False)
    
    scale=1

    v0 = Visualizer(
            img_rubin,
            metadata=astrotest_metadata,
            scale=scale,
            instance_mode=ColorMode.SEGMENTATION, # remove the colors of unsegmented pixels. This option is only available for segmentation models
            keypoint_threshold=0
    )


    out = v0.draw_instance_predictions(outputs['instances'].to("cpu"),lf=False,ls='-',alpha=0.1,boxf=True, custom_labels=np.arange(len(outputs['instances'])))
    print("detected instances:", len(outputs['instances'].pred_boxes))
    print("")
    
    ax0.imshow(out.get_image())
    ax0.axis("off")
    ax0.set_title('Rubin')

    ###############################################
    v1 = Visualizer(
                img_roman,
                metadata=astrotest_metadata,
                scale=scale,
                instance_mode=ColorMode.SEGMENTATION, # remove the colors of unsegmented pixels. This option is only available for segmentation models
                keypoint_threshold=0
    )


    out = v1.draw_instance_predictions(outputs['instances'].to("cpu"),lf=False,ls='-',alpha=0.01,boxf=True, custom_labels=np.arange(len(outputs['instances'])))

    ax1.imshow(out.get_image())
    #ax1.imshow(img_roman)
    ax1.axis("off")
    ax1.set_title('Roman')

    #cs= outputs['instances'].pred_boxes.get_centers().detach().numpy()
    #ax2.scatter(cs[:,0],cs[:,1],marker='x',color='purple',s=70)

    plt.tight_layout()

    plt.show()

In [ ]:
#b = next(iter(train_loader))
i1 = np.random.randint(len(test_dicts))
i2=i1
while i2==i1:
    i2 = np.random.randint(len(test_dicts))

d1 = test_dicts[i1]
md1 = mapper(d1)
outputs = model([md1])

d2 = test_dicts[i2]
md2 = mapper(d2)
plot_images_romancomp(md1,md2,d1,outputs[0],roman=True)

In [ ]:
600/12

## Losses

In [ ]:
losses30k = np.load('/work/hdd/bdsp/yse2/lsst_runs/lsst5_30k_4h200_bs192_ep50/lsst5_30k_4h200_bs192_ep50_losses.npy')
#with open('/work/hdd/bdsp/yse2/lsst_runs/clip5_30k_4h200_bs64_ep50/clip_30k_4h200_bs64_ep50_losses.json') as f:
#    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')
'''
ckey = 'roi_contrastive_loss'

closses = []
keylosses = []
temp=[]
iters = []
dlosses = []

for iteration, loss in losses.items():
    iteration = int(iteration)
    closs = loss[ckey]
    dloss=0
    for k,l in loss.items():
        if k!=ckey and k!='logit_scale':
            dloss+=l
    keylosses.append(loss['loss_keypoint'])
    iters.append(iteration)
    closses.append(closs)
    dlosses.append(dloss)
'''

fig,ax = plt.subplots(1,2,figsize=(10,5))

ax[0].plot(losses30k)
ax[0].legend(loc=0)
ax[0].set_ylim(0,15)


ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Loss')
#plt.axhline(np.log(8),color='k',linestyle='--')

ax[0].axvline(len(losses30k)*15/50,color='k')
ax[0].axvline(len(losses30k)*25/50,color='k')
ax[0].axvline(len(losses30k)*35/50,color='k')


#ax[0].plot(dlosses[::3],alpha=0.3)

#ax[1].set_ylim(3,8)
#ax[1].axvline(len(losses)*15/50,color='k')
#ax[1].axvline(len(losses)*25/50,color='k')
#ax[1].axvline(len(losses)*35/50,color='k')

ax[0].set_title('Baseline')
#ax[1].set_title('Clip')

ax[0].axhline(5.79)

ax[0].set_xlim(0,len(losses30k))

In [ ]:
l=np.array([0.07299594581127167, 0.050019271671772, 0.18006189167499542, 0.1008935272693634, 0.16175025701522827, 0.11292232573032379, 0.38743746280670166, 0.13874311745166779, 0.4041510820388794, 0.19807933270931244, 3.9884414672851562])
l.sum()

In [ ]:
with open('/work/hdd/bdsp/yse2/lsst_runs/clip5_30k_4h200_bs64_ep50/clip5_30k_4h200_bs64_ep50_losses.json') as f:
    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')

ckey = 'roi_contrastive_loss'

closses = []
keylosses = []
temp=[]
iters = []
dlosses = []
for iteration, loss in losses.items():
    iteration = int(iteration)
    closs = loss[ckey]
    dloss=0
    for k,l in loss.items():
        if k!=ckey and k!='logit_scale':
            dloss+=l
    keylosses.append(loss['loss_keypoint'])
    iters.append(iteration)
    closses.append(closs)
    dlosses.append(dloss)
    temp.append(loss['logit_scale'])

plt.plot(iters,closses,label='contrastive loss')
plt.plot(iters,dlosses,label='total detection loss')
plt.plot(iters,keylosses,label='keypoint loss')

plt.legend(loc=0)


plt.ylim(0,15)
#loss2 = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_moco5_30k_rubin_test/only_moco5_30k_rubin_test_losses.npy')
#plt.plot(loss)
plt.xlabel('Iteration')
plt.ylabel('Loss')
#plt.axhline(np.log(8),color='k',linestyle='--')

plt.axvline(len(losses)*15/50,color='k')
plt.axvline(len(losses)*25/50,color='k')
plt.axvline(len(losses)*35/50,color='k')

plt.xlim(-10,len(losses)/50*50)
plt.axhline(4.75)
plt.axhline(7.0)

plt.title('Old')

In [ ]:
def dist_losses(losses):
    distkey = 'loss_distillation'
    distlosses = []
    keylosses = []
    temp=[]
    iters = []
    dlosses = []
    for iteration, loss in losses.items():
        iteration = int(iteration)
        dloss=0
        for k,l in loss.items():
            dloss+=l
        keylosses.append(loss['loss_keypoint'])
        iters.append(iteration)
        if distkey in loss:
            distlosses.append(loss[distkey])
        dlosses.append(dloss)
    return iters, dlosses, keylosses,distlosses

with open('/work/hdd/bdsp/yse2/lsst_runs/distill_30k_4h200_bs192_ep50/distill_30k_4h200_bs192_ep50_losses.json') as f:
    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')

with open('/work/hdd/bdsp/yse2/lsst_runs/distill_30k_4h200_bs192_ep50/distill_30k_4h200_bs192_ep50_val_losses.json') as f:
    vlosses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')



iters, dlosses, keylosses, distlosses = dist_losses(losses)
itersv, dlossesv, keylossesv, distlossesv = dist_losses(vlosses)

fig,ax = plt.subplots(1,2,figsize=(10,5))
#ax[0].plot(iters,closses,label='contrastive loss')
ax[0].plot(iters,dlosses,label='total detection loss')
ax[0].plot(iters,keylosses,label='keypoint loss')
ax[0].legend(loc=0)
ax[0].set_ylim(0,15)
#plt.plot(loss)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Loss')
ax[0].axhline(4.5)
ax[0].axhline(3.2)



ax[1].plot(itersv,dlossesv,label='total detection loss')
ax[1].plot(itersv,keylossesv,label='keypoint loss')
ax[1].legend(loc=0)
ax[1].set_ylim(0,15)
#plt.plot(loss)
ax[1].set_xlabel('Iteration')
ax[1].set_ylabel('Loss')
ax[1].axhline(4.5)
ax[1].axhline(3.2)

In [ ]:
vlosses['156']

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,5))
#ax[0].plot(iters,closses,label='contrastive loss')
ax.plot(iters,distlosses,label='distillation loss')
ax.legend(loc=0)
#ax.set_ylim(0,15)
#plt.plot(loss)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
#ax.axhline(4.5)
#ax.axhline(3.2)




In [ ]:
with open('/work/hdd/bdsp/yse2/lsst_runs/clip5_30k_4h200_bs64_ep50/clip_30k_4h200_bs64_ep50_new_losses.json') as f:
    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')

ckey = 'roi_contrastive_loss'

closses = []
keylosses = []
temp=[]
iters = []
dlosses = []

for iteration, loss in losses.items():
    iteration = int(iteration)
    closs = loss[ckey]
    dloss=0
    for k,l in loss.items():
        if k!=ckey and k!='logit_scale':
            dloss+=l
    keylosses.append(loss['loss_keypoint'])
    iters.append(iteration)
    closses.append(closs)
    dlosses.append(dloss)

plt.plot(iters,closses,label='contrastive loss')
plt.plot(iters,dlosses,label='total detection loss')
plt.plot(iters,keylosses,label='keypoint loss')

plt.legend(loc=0)


plt.ylim(0,8)
#loss2 = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_moco5_30k_rubin_test/only_moco5_30k_rubin_test_losses.npy')
#plt.plot(loss)
plt.xlabel('Iteration')
plt.ylabel('Loss')
#plt.axhline(np.log(8),color='k',linestyle='--')

plt.axvline(len(losses)*15/50,color='k')
plt.axvline(len(losses)*25/50,color='k')
plt.axvline(len(losses)*35/50,color='k')

#plt.xlim(-10,600)

In [ ]:
with open('/work/hdd/bdsp/yse2/lsst_runs/distill_30k_4h200_bs192_ep50/distill_30k_4h200_bs192_ep50_losses.json') as f:
    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')

keys = ['loss_cls_stage0', 'loss_box_reg_stage0', 'loss_cls_stage1', 'loss_box_reg_stage1', 
        'loss_cls_stage2', 'loss_box_reg_stage2', 'loss_mask', 'loss_rpn_cls', 'loss_rpn_loc']

#ckey = 'roi_contrastive_loss'
#closses = []
keylosses = []
temp=[]
iters = []
dlosses = []
ldict = {}

for k in keys:
    ldict[k] = [0]

for iteration, loss in losses.items():
    iteration = int(iteration)
    #closs = loss[ckey]
    dloss=0
    for k,l in loss.items():
        if k!=ckey and k!='logit_scale':
            dloss+=l
        if k in keys:
            li = ldict[k]
            li.append(l)
            ldict[k] = li
    keylosses.append(loss['loss_keypoint'])
    iters.append(iteration)
    
    dlosses.append(dloss)
    #closses.append(closs)
fig = plt.figure(figsize=(10,5))
#plt.plot(iters,closses,label='contrastive loss')
for ki in range(len(keys)):
    plt.plot(iters,ldict[keys[ki]][1:],label=keys[ki])
#plt.plot(iters,dlosses,label='total detection loss')
#plt.plot(iters,keylosses,label='keypoint loss')

plt.legend(loc=0)


#plt.ylim(0,15)
#loss2 = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_moco5_30k_rubin_test/only_moco5_30k_rubin_test_losses.npy')
#plt.plot(loss)
plt.xlabel('Iteration')
plt.ylabel('Loss')
#plt.axhline(np.log(8),color='k',linestyle='--')

plt.axvline(len(losses)*15/50,color='k')
plt.axvline(len(losses)*25/50,color='k')
plt.axvline(len(losses)*35/50,color='k')

plt.ylim(0,0.5)

In [ ]:
def return_losses(losses,ckey = 'roi_contrastive_loss'):
    closses = []
    keylosses = []
    temp=[]
    iters = []
    dlosses = []
    #for iteration, loss in losses.items():
    for iteration in np.arange(1,len(losses)+1):
        loss = losses[str(iteration)]
        #iteration = int(iteration)
        closs = loss[ckey]
        dloss=0
        for k,l in loss.items():
            if k!=ckey and k!='logit_scale':
                dloss+=l
        keylosses.append(loss['loss_keypoint'])
        iters.append(iteration)
        closses.append(closs)
        dlosses.append(dloss)
        temp.append(loss['logit_scale'])
    return iters,keylosses,closses,dlosses

with open('/work/hdd/bdsp/yse2/lsst_runs/clip5_flatten_30k_4h200_bs64_ep15_resume/clip5_flatten_30k_4h200_bs64_ep15_total_losses.json') as f:
    new_losses = json.load(f)

iters,keylosses,closses,dlosses = return_losses(new_losses)

with open('/work/hdd/bdsp/yse2/lsst_runs/clip5_30k_4h200_bs64_ep50/clip5_30k_4h200_bs64_ep50_losses.json') as f:
    old_losses = json.load(f)

iters,old_keylosses,old_closses,old_dlosses = return_losses(old_losses)

fig,ax = plt.subplots(1,2,figsize=(10,5))
ax[0].plot(iters,closses,label='contrastive loss')
ax[0].plot(iters,dlosses,label='total detection loss')
ax[0].plot(iters,keylosses,label='keypoint loss')
ax[0].set_title('New')

ax[1].plot(iters,old_closses,label='contrastive loss')
ax[1].plot(iters,old_dlosses,label='total detection loss')
ax[1].plot(iters,old_keylosses,label='keypoint loss')
ax[1].set_title('Old')


for axi in ax:
    axi.legend(loc=0)
    axi.set_ylim(0,15)
    axi.set_xlabel('Iteration')
    axi.set_ylabel('Loss')

    axi.axhline(4.5)
    axi.axhline(3.2)
    axi.axhline(4.5)
    axi.axhline(3.2)
#ax[0].set_xlim(0,500)

In [ ]:
with open('/work/hdd/bdsp/yse2/lsst_runs/test_distill/test_distill_losses.json') as f:
    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')


ckey = 'roi_contrastive_loss'

closses = []
keylosses = []
temp=[]
iters = []
dlosses = []
for iteration, loss in losses.items():
    iteration = int(iteration)
    #closs = loss[ckey]
    dloss=0
    for k,l in loss.items():
        #if k!=ckey and k!='logit_scale':
        dloss+=l
    keylosses.append(loss['loss_keypoint'])
    iters.append(iteration)
    #closses.append(closs)
    dlosses.append(dloss)
    #temp.append(loss['logit_scale'])


fig,ax = plt.subplots(1,2,figsize=(10,5))
#ax[0].plot(iters,closses,label='contrastive loss')
ax[0].plot(iters,dlosses,label='total detection loss')
ax[0].plot(iters,keylosses,label='keypoint loss')

ax[0].legend(loc=0)


ax[0].set_ylim(0,15)
#loss2 = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_moco5_30k_rubin_test/only_moco5_30k_rubin_test_losses.npy')
#plt.plot(loss)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Loss')
#plt.axhline(np.log(8),color='k',linestyle='--')
ax[1].set_ylabel('Temp')

ax[1].plot(temp)
ax[0].axhline(4.75)
ax[0].axhline(7.0)

In [ ]:
with open('/work/hdd/bdsp/yse2/lsst_runs/comb_30k_4h200_bs144_ep50/comb_30k_4h200_bs144_ep50_losses.json') as f:
    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')


ckey = 'roi_contrastive_loss'

closses = []
keylosses = []
temp=[]
iters = []
dlosses = []
for iteration, loss in losses.items():
    iteration = int(iteration)
    #closs = loss[ckey]
    dloss=0
    for k,l in loss.items():
        #if k!=ckey and k!='logit_scale':
        dloss+=l
    keylosses.append(loss['loss_keypoint'])
    iters.append(iteration)
    #closses.append(closs)
    dlosses.append(dloss)
    #temp.append(loss['logit_scale'])


fig,ax = plt.subplots(1,2,figsize=(10,5))
#ax[0].plot(iters,closses,label='contrastive loss')
ax[0].plot(iters,dlosses,label='total detection loss')
ax[0].plot(iters,keylosses,label='keypoint loss')

ax[0].legend(loc=0)


ax[0].set_ylim(0,15)
#loss2 = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_moco5_30k_rubin_test/only_moco5_30k_rubin_test_losses.npy')
#plt.plot(loss)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Loss')
#plt.axhline(np.log(8),color='k',linestyle='--')
ax[1].set_ylabel('Temp')

ax[1].plot(temp)
ax[0].axhline(4.5)
ax[0].axhline(3.2)

In [ ]:
with open('/work/hdd/bdsp/yse2/lsst_runs/test_fixtemp/test_fixtemp_losses.json') as f:
    losses = json.load(f)
    #loss = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_rubin_30k_clip_test_maxpool/only_rubin_30k_clip_test_maxpool_losses.npy')


ckey = 'roi_contrastive_loss'

closses = []
keylosses = []
temp=[]
iters = []
dlosses = []
for iteration, loss in losses.items():
    iteration = int(iteration)
    closs = loss[ckey]
    dloss=0
    for k,l in loss.items():
        if k!=ckey and k!='logit_scale':
            dloss+=l
    keylosses.append(loss['loss_keypoint'])
    iters.append(iteration)
    closses.append(closs)
    dlosses.append(dloss)
    temp.append(loss['logit_scale'])


fig,ax = plt.subplots(1,2,figsize=(10,5))
ax[0].plot(iters,closses,label='contrastive loss')
ax[0].plot(iters,dlosses,label='total detection loss')
ax[0].plot(iters,keylosses,label='keypoint loss')

ax[0].legend(loc=0)

ax[0].axhline(4.45)

ax[0].set_ylim(0,15)
#loss2 = np.load('/work/hdd/bdsp/yse2/lsst_runs/only_moco5_30k_rubin_test/only_moco5_30k_rubin_test_losses.npy')
#plt.plot(loss)
ax[0].set_xlabel('Iteration')
ax[0].set_ylabel('Loss')
#plt.axhline(np.log(8),color='k',linestyle='--')
ax[1].set_ylabel('Temp')
ax[1].plot(temp)

In [ ]:
#b = next(iter(train_loader))
i1 = np.random.randint(len(test_dicts))
i2=i1
while i2==i1:
    i2 = np.random.randint(len(test_dicts))
b1 = [mapper(test_dicts[i1])]
b2 = [mapper(test_dicts[i2])]

In [ ]:
b1[0].keys()

In [ ]:
192/4

In [ ]:
import time
from deepdisc.astrodet.visualizer import Visualizer
from deepdisc.astrodet.visualizer import ColorMode
from detectron2 import structures
from astropy.visualization import make_lupton_rgb

def plot_images_annos(d,ddict,resize=False,plot_truth=False,buffer=1):

    if resize:
        image = d['image']        
        b1 = image[-1]
        b2 = image[-2]
        b3 = image[-3]
        img_roman = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=15, Q=5)

    else:
        image = d['image_rubin']

    # For LSST
    b1 = image[3]
    b2 = image[2]
    b3 = image[1]
    img_rubin = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=0.5, Q=5)

    if resize:
        fig,ax = plt.subplots(1,2,figsize=(10,5))
        ax0 = ax[0]
        ax1=ax[1]

    else:
        fig,ax = plt.subplots(1,1,figsize=(5,5))
        ax0 = ax
    if resize:
        scale=1
    else:
        scale=5
    print("total instances: ", len(d["instances"]))
    v0 = Visualizer(
        img_rubin,
        metadata=astrotest_metadata,
        scale=scale,
        instance_mode=ColorMode.SEGMENTATION,  # remove the colors of unsegmented pixels. This option is only available for segmentation models
    )

    if resize:
        groundTruth = v0.draw_dataset_dict(d, lf=False, alpha=0.01)
        ax0.imshow(groundTruth.get_image())

    else:
        #groundTruth = v0.draw_dataset_dict(ddict, lf=False, alpha=0.01)
        #ax0.imshow(groundTruth.get_image())
        ax0.imshow(img_rubin)
        xs = [a['keypoints'][0] for a in ddict['annotations']]
        ys = [a['keypoints'][1] for a in ddict['annotations']]

        ax0.scatter(xs,ys,color='w',s=10)

    ax0.axis("off")
    if resize:
        v1 = Visualizer(
            img_roman,
            metadata=astrotest_metadata,
            scale=scale,
            instance_mode=ColorMode.SEGMENTATION,  # remove the colors of unsegmented pixels. This option is only available for segmentation models
        )
        
        groundTruth = v1.draw_dataset_dict(d, lf=False, alpha=0.01)
        #groundTruth = v0.draw_instance_predictions(d['instances'],lf=False,ls='-',alpha=0.1,boxf=True, custom_labels=np.arange(len(outputs['instances'])))
    
        #plt.figure(figsize=(12,6))
    
        ax1 = ax[1]
        ax1.imshow(groundTruth.get_image())
        ax1.axis("off")
    

    #ax0.sc

    plt.tight_layout()

    plt.show()

In [ ]:
mapper_baseline =cfg.dataloader.train.mapper(cfg.dataloader.train.keypoint_hflip_indices,cfg.dataloader.train.imagereader, lsst_key_mapper, None).map_data # No augs for test set

mapper_cv2 = ResizeCombinedMapper(cfg.dataloader.train.keypoint_hflip_indices,cfg.dataloader.train.imagereader, lsst_key_mapper, cfg.dataloader.augs).map_data # No augs for test set
#mapper_interp = ResizeCombinedMapper(cfg.dataloader.train.keypoint_hflip_indices,'reproj_interp',cfg.dataloader.train.imagereader, lsst_key_mapper, cfg.dataloader.augs).map_data # No augs for test set
#mapper_gaus = ResizeCombinedMapper(cfg.dataloader.train.keypoint_hflip_indices,'reproj_gaussian',cfg.dataloader.train.imagereader, lsst_key_mapper, cfg.dataloader.augs).map_data # No augs for test set
#mapper_hann = ResizeCombinedMapper(cfg.dataloader.train.keypoint_hflip_indices,'reproj_hann',cfg.dataloader.train.imagereader, lsst_key_mapper, cfg.dataloader.augs).map_data # No augs for test set


In [ ]:
j=107
dm=mapper_cv2(test_dicts[j])
dm_base = mapper_baseline(test_dicts[j])
plot_images_annos(dm,test_dicts[j],resize=True)

In [ ]:
12*9*512*512*4/1e9

In [ ]:


print('Single image:',6*160*160*4/1e6,'Mb')
print('Batch Size 48:', 48*6*160*160*4/1e6,'Mb')

In [ ]:


print('Single image:',9*512*512*4/1e6,'Mb')
print('Batch Size 48:', 48*9*512*512*4/1e6,'Mb')
print('Batch Size 12:', 12*9*512*512*4/1e6,'Mb')

In [ ]:
39498.346576690674*1.6/3600

In [ ]:
144/4

In [ ]:
48*3

In [ ]:
140/40

In [ ]:
data_root_dir = '/work/hdd/bdsp/yse2/lsst_data/'
anns_folder = 'annotations_lvl5'
test_data_fn = f'{data_root_dir}{anns_folder}/test_8k_keypoints_wcs.json'

cfgfile = "/work/hdd/bdsp/g4merz/RubinRoman/configs/solo/swin_distill_job_keypoint_30k.py"
#cfgfile = "/work/hdd/bdsp/g4merz/RubinRoman/configs/solo/swin_lsst_job_keypoint_30k.py"

bs=1
cfg = LazyConfig.load(cfgfile)
cfg.SEED = -1
cfg.dataloader.train.cache_dir='/work/hdd/bfhm/g4merz/wcs_map_cache/test_8k_keypoints_wcs'


In [ ]:
from detectron2.data import MetadataCatalog, DatasetCatalog

custom_colors = [
    (0, 255, 0),    # green for galaxies
    (0, 0, 255),    # blue for stars
]

custom_prop_colors = [
    (255,255,255), # white for foreground for background
    (255, 0, 0)    
    
]

if cfg.DATASETS.TEST in MetadataCatalog:
    DatasetCatalog.remove(cfg.DATASETS.TEST)
    MetadataCatalog.remove(cfg.DATASETS.TEST)

if cfg.DATASETS.TRAIN in MetadataCatalog:
    DatasetCatalog.remove(cfg.DATASETS.TRAIN)
    MetadataCatalog.remove(cfg.DATASETS.TRAIN)
    
if "astro_prop" in MetadataCatalog:
    DatasetCatalog.remove("astro_prop")
    MetadataCatalog.remove("astro_prop")

astrotest_metadata = register_data_set(cfg.DATASETS.TEST, test_data_fn, thing_classes=cfg.metadata.classes, thing_colors=custom_colors)
astrotest_metadata = register_data_set(cfg.DATASETS.TRAIN, test_data_fn, thing_classes=cfg.metadata.classes, thing_colors=custom_colors)

astroprop_metadata = register_data_set("astro_prop", test_data_fn, thing_classes=["background","foreground"], thing_colors=custom_prop_colors)

In [ ]:
import os
def lsst_key_mapper(dataset_dict):
    key = dataset_dict["file_name"]
    im = key.split('truth/')[1]
    k = os.path.join("/work/hdd/bdsp/yse2/lsst_data/truth/",im)
    return k
mapper = cfg.dataloader.train.mapper(cfg.dataloader.train.keypoint_hflip_indices,cfg.dataloader.train.cache_dir,
                                     cfg.dataloader.train.imagereader, lsst_key_mapper, None).map_data # No augs for test set
#test_loader = return_test_loader(cfg, mapper)
train_loader = return_train_loader(cfg, mapper)

In [ ]:
test_dicts = get_data_from_json(test_data_fn)

In [ ]:
#b = next(iter(train_loader))
i1 = np.random.randint(len(test_dicts))
i2=i1
while i2==i1:
    i2 = np.random.randint(len(test_dicts))
dm1 = [mapper(test_dicts[i1])]
dm2 = [mapper(test_dicts[i2])]

In [ ]:
dm1[0].keys()

In [ ]:
fig,ax = plt.subplots(3,3,figsize=(10,10))
for i,ax in enumerate(ax.ravel()):
    ax.imshow(dm1[0]['image_combined'][i],norm='asinh')
    #ax.axis('off')
plt.tight_layout()

In [ ]:
sys.path.append('/work/hdd/bdsp/g4merz/RubinRoman/configs/')
from custom.mappers import CLIPMapper
from custom.image_readers import  DualRomanRubinImageReader

In [ ]:
import os
def lsst_key_mapper(dataset_dict):
    key = dataset_dict["file_name"]
    im = key.split('truth/')[1]
    k = os.path.join("/work/hdd/bdsp/yse2/lsst_data/truth/",im)
    return k

#mapper = cfg.dataloader.test.mapper(cfg.dataloader.test.keypoint_hflip_indices,cfg.dataloader.test.cache_dir,
#                                     cfg.dataloader.test.imagereader, lsst_key_mapper, None).map_data # No augs for test set
#mapper = cfg.dataloader.test.mapper(cfg.dataloader.test.keypoint_hflip_indices,
#                                     cfg.dataloader.test.imagereader, lsst_key_mapper, None).map_data # No augs for test set

mapper = CLIPMapper([0], DualRomanRubinImageReader(), lsst_key_mapper,None).map_data


In [ ]:
test_dicts = get_data_from_json(test_data_fn)

In [ ]:
from astropy import units as u
wcsi = WCS(test_dicts[3]['wcs_roman'])
wcsi.proj_plane_pixel_scales()[0].to(u.arcsec)

In [ ]:
0.2/0.0575

In [ ]:
m1 = mapper(test_dicts[3])

In [ ]:
m1['image_rubin'].shape

In [ ]:
from astropy.visualization import make_lupton_rgb



image_roman = m1['image_roman']        
b1 = image_roman[-1]
b2 = image_roman[-2]
b3 = image_roman[-3]
img_roman = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=15, Q=5)

image_rubin = m1['image_rubin']

# For LSST
b1 = image_rubin[3]
b2 = image_rubin[2]
b3 = image_rubin[1]
img_rubin = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=0.5, Q=5)


In [ ]:
0.2

In [ ]:

from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
fontprops = fm.FontProperties(size=15)
fig,ax = plt.subplots(1,1,figsize=(5,5))
scalebar = AnchoredSizeBar(ax.transData,
                           50, '10"', 'upper left', 
                           pad=0.1,
                           color='white',
                           frameon=False,
                           size_vertical=2,
                           fontproperties=fontprops)
ax.add_artist(scalebar)
ax.imshow(img_rubin)
#ax.axis('off')
ax.set_xlabel('x',fontsize=14)
ax.set_ylabel('y',fontsize=14)

plt.savefig('/work/hdd/bdsp/g4merz/RubinRoman/plots/ex_rubin_img.png',bbox_inches='tight')


In [ ]:
fontprops = fm.FontProperties(size=15)
fig,ax = plt.subplots(1,1,figsize=(5,5))
scalebar = AnchoredSizeBar(ax.transData,
                           174, '10"', 'upper left', 
                           pad=0.1,
                           color='white',
                           frameon=False,
                           size_vertical=7,
                           fontproperties=fontprops)
ax.add_artist(scalebar)
ax.imshow(img_roman)
#ax.axis('off')
ax.set_xlabel('x',fontsize=14)
ax.set_ylabel('y',fontsize=14)

plt.savefig('/work/hdd/bdsp/g4merz/RubinRoman/plots/ex_roman_img.png',bbox_inches='tight')


In [ ]:
2*(0.2/0.0575)

In [ ]:
(0.2/0.0575)

In [ ]:
#ax[1].imshow(img_roman)